# 01 — CNN Standalone Model

Custom 4-layer CNN trained on 224×224 MRI images for 4-class brain tumor classification.
Saves `cnn_model.h5` to `../saved_models/` for use in 02_CNN_Ensemble.ipynb and 08_AllModelsCombined.ipynb.

## Section 0: Google Colab Setup & Dataset Download

**Run this cell first every time you open a new Colab session.**

### One-time Colab Secrets setup
Go to **Runtime → Manage secrets** and add these three secrets:

| Secret name | Value |
|---|---|
| `KAGGLE_USERNAME` | sk1285 |
| `KAGGLE_KEY` | (your kaggle API key from kaggle.com/settings/account) |
| `HF_TOKEN` | (your HuggingFace token from huggingface.co/settings/tokens) |

Once secrets are saved they persist across all Colab sessions — you only do this once.

### What this cell does
- Installs `kaggle` and `huggingface_hub`
- Reads credentials from Colab Secrets
- Downloads the Brain Tumor MRI dataset once to `/content/MRI_DATASET/`  
  (all 8 notebooks share the same folder — subsequent notebooks skip the download)
- Sets path variables used by later cells

In [ ]:
import sys, os, json

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    os.system("pip install huggingface_hub -q")
    from google.colab import drive, userdata

    HF_TOKEN = userdata.get('HF_TOKEN')   # Colab Secret: HF_TOKEN

    print("Mounting Google Drive...")
    drive.mount('/content/drive')

    # Dataset lives in Google Drive.
    # One-time setup: open the link below, click "Add shortcut to Drive",
    # place it in "My Drive" and name it exactly  MRI_DATASET
    # https://drive.google.com/drive/folders/15cP-SVH3BT20ogDjuwS5tXnVuXIW9Doe
    DATASET_PATH     = "/content/drive/MyDrive/MRI_DATASET/"
    SAVED_MODELS_DIR = "/content/saved_models/"
    RESULTS_DIR      = "/content/results/"

    if not os.path.isdir(DATASET_PATH + "Training"):
        raise RuntimeError(
            "Dataset not found at " + DATASET_PATH + "\n"
            "Fix:\n"
            "  1. Open: https://drive.google.com/drive/folders/15cP-SVH3BT20ogDjuwS5tXnVuXIW9Doe\n"
            "  2. Click 'Add shortcut to Drive' → My Drive\n"
            "  3. Name the shortcut exactly:  MRI_DATASET\n"
            "  4. Re-run this cell"
        )

else:
    HF_TOKEN     = os.environ.get('HF_TOKEN', '')
    DATASET_PATH     = "../MRI_DATASET/"
    SAVED_MODELS_DIR = "../saved_models/"
    RESULTS_DIR      = "../results/"

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Print actual folder names found so you can verify they match CLASS_NAMES
_train_path = os.path.join(DATASET_PATH, "Training")
_test_path  = os.path.join(DATASET_PATH, "Testing")
_train_cls  = sorted([d for d in os.listdir(_train_path) if os.path.isdir(os.path.join(_train_path, d))])
_test_cls   = sorted([d for d in os.listdir(_test_path)  if os.path.isdir(os.path.join(_test_path,  d))])
print(f"  Dataset path     : {DATASET_PATH}")
print(f"  Training folders : {_train_cls}")
print(f"  Testing  folders : {_test_cls}")
for _c in _train_cls:
    _imgs = [f for f in os.listdir(os.path.join(_train_path, _c)) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f"    Training/{_c}: {len(_imgs)} images")
print("Environment ready")


## Section 1: Imports & Configuration

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize
print("\u2713 Imports complete")

## Section 2: Constants & Hyperparameters

In [ ]:
NOTEBOOK_NAME = "01_CNN_Standalone"

# Dataset
DATASET_PATH     = globals().get("DATASET_PATH", "../MRI_DATASET/")
TRAIN_DIR        = DATASET_PATH + "Training/"
TEST_DIR         = DATASET_PATH + "Testing/"

# Classes — FIXED ORDER, DO NOT CHANGE
CLASS_NAMES      = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES      = 4

# Image settings
IMG_HEIGHT       = 224
IMG_WIDTH        = 224
CHANNELS         = 3

# Training hyperparameters
BATCH_SIZE       = 32
EPOCHS           = 20
LEARNING_RATE    = 1e-4
VALIDATION_SPLIT = 0.2

# Reproducibility
RANDOM_SEED      = 42

# Paths
SAVED_MODELS_DIR = globals().get("SAVED_MODELS_DIR", "../saved_models/")
RESULTS_DIR      = globals().get("RESULTS_DIR", "../results/")
RESULTS_NB_DIR   = os.path.join(RESULTS_DIR, NOTEBOOK_NAME)
HF_TOKEN         = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN", ""))
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_NB_DIR, exist_ok=True)

# Set seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

print("\u2713 Constants configured")
print(f"  Image size : {IMG_HEIGHT}\u00d7{IMG_WIDTH}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Epochs     : {EPOCHS}")
print(f"  Classes    : {CLASS_NAMES}")

In [ ]:
# Hard stop if folder names do not match CLASS_NAMES
_found = sorted([d for d in os.listdir(TRAIN_DIR)
                 if os.path.isdir(os.path.join(TRAIN_DIR, d))])

if _found != sorted(CLASS_NAMES):
    raise ValueError(
        f"Folder names do not match CLASS_NAMES!\n"
        f"  Expected : {sorted(CLASS_NAMES)}\n"
        f"  Found    : {_found}\n"
        f"Update CLASS_NAMES in the constants cell to exactly match the folder names above."
    )

_total_train = sum(len([f for f in os.listdir(os.path.join(TRAIN_DIR, c))
                         if f.lower().endswith(('.jpg','.jpeg','.png'))])
                   for c in CLASS_NAMES)
_total_test  = sum(len([f for f in os.listdir(os.path.join(TEST_DIR, c))
                         if f.lower().endswith(('.jpg','.jpeg','.png'))])
                   for c in CLASS_NAMES)

if _total_train == 0:
    raise RuntimeError("Training directory is empty — dataset not downloaded correctly.")

print("Dataset verified")
print(f"  Training images : {_total_train}  (~{_total_train // len(CLASS_NAMES)} per class)")
print(f"  Testing  images : {_total_test}   (~{_total_test  // len(CLASS_NAMES)} per class)")
print(f"  Class → index   : {list(enumerate(CLASS_NAMES))}")
print("  Class order is fixed — the webapp uses the same mapping.")


In [ ]:
# ── Dataset path check — will raise immediately if folder names do not match ─
_expected = CLASS_NAMES  # ['glioma', 'meningioma', 'notumor', 'pituitary']
_found    = sorted([d for d in os.listdir(TRAIN_DIR)
                    if os.path.isdir(os.path.join(TRAIN_DIR, d))])

if _found != _expected:
    raise ValueError(
        f"Folder names do not match CLASS_NAMES!\n"
        f"  Expected : {_expected}\n"
        f"  Found    : {_found}\n"
        f"Update CLASS_NAMES in the constants cell to match the actual folder names."
    )

_total_train = sum(
    len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in CLASS_NAMES
)
_total_test  = sum(
    len(os.listdir(os.path.join(TEST_DIR, c))) for c in CLASS_NAMES
)

if _total_train == 0:
    raise RuntimeError("Training directory is empty — dataset was not downloaded correctly.")

print("✓ Dataset verified")
print(f"  Training images : {_total_train}  ({_total_train // len(CLASS_NAMES)} avg per class)")
print(f"  Testing  images : {_total_test}  ({_total_test  // len(CLASS_NAMES)} avg per class)")
print(f"  Class order     : {list(zip(range(len(CLASS_NAMES)), CLASS_NAMES))}")
print("  This order is used by the model — do not change CLASS_NAMES.")


## Section 3: Data Loading & Verification

In [ ]:
# --- Augmentation for training data ---
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=VALIDATION_SPLIT
)

# --- No augmentation for test/validation data ---
test_datagen = ImageDataGenerator(rescale=1./255)

# --- Training generator ---
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='training',
    seed=RANDOM_SEED,
    shuffle=True
)

# --- Validation generator ---
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='validation',
    seed=RANDOM_SEED,
    shuffle=False
)

# --- Test generator ---
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

# --- Verification ---
print("=" * 50)
print("DATA VERIFICATION")
print(f"Class indices      : {train_generator.class_indices}")
print(f"Training samples   : {train_generator.samples}")
print(f"Validation samples : {val_generator.samples}")
print(f"Test samples       : {test_generator.samples}")
print(f"Image size         : {IMG_HEIGHT}\u00d7{IMG_WIDTH}")
print(f"Batch size         : {BATCH_SIZE}")
print("=" * 50)

## Section 4: Data Preprocessing & Augmentation

In [ ]:
# Augmentation is defined in the ImageDataGenerator in Section 3.
# No additional preprocessing needed — rescale to [0,1] and augment on the fly.
print("\u2713 Data augmentation configured via ImageDataGenerator")

## Section 5: Model Definition

In [ ]:
# --- Custom CNN architecture ---
# 4\u00d7 (Conv2D \u2192 MaxPooling), then Dense feature layer, Dropout, softmax output
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu', name='feature_layer'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax', name='output_layer'),
], name='cnn_model')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print("\u2713 Model defined and compiled")

## Section 6: Model Training

In [ ]:
# --- Callbacks ---
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        filepath=SAVED_MODELS_DIR + 'cnn_model_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# --- Train ---
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\u2713 Training complete")

# --- Plot training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

_fig_path = os.path.join(RESULTS_NB_DIR, "training_history.jpg")
plt.suptitle(f'{NOTEBOOK_NAME} \u2014 Training History')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_NB_DIR, f"{NOTEBOOK_NAME}_training_history.jpg"),
            dpi=150, bbox_inches='tight')
plt.show()

## Section 7: Model Evaluation

In [ ]:
import re as _re

def evaluate_model(model, generator, model_name="Model"):
    """Standard evaluation: confusion matrix, classification report, ROC, PR curves."""
    _fname = _re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    generator.reset()
    y_pred_proba = model.predict(generator, verbose=1)
    y_pred       = np.argmax(y_pred_proba, axis=1)
    y_true       = generator.classes

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_name} — Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_confusion_matrix.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    # Classification Report
    print(f"\n{model_name} — Classification Report")
    print("=" * 60)
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    # ROC Curve
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3])
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} — ROC Curve')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_roc_curve.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    # Precision-Recall Curve
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_proba[:, i])
        ap = average_precision_score(y_true_bin[:, i], y_pred_proba[:, i])
        plt.plot(recall, precision, label=f'{cls} (AP = {ap:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'{model_name} — Precision-Recall Curve')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_pr_curve.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    return y_pred, y_pred_proba

# Evaluate on test set
y_pred, y_pred_proba = evaluate_model(model, test_generator, model_name=NOTEBOOK_NAME)
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"\n✓ Test Accuracy : {test_acc:.4f}")
print(f"✓ Test Loss     : {test_loss:.4f}")

## Section 8: Save Model

In [ ]:
# --- Save final model ---
model_path = SAVED_MODELS_DIR + 'cnn_model.h5'
model.save(model_path)
print(f"\u2713 Model saved to: {model_path}")
print("  This model is used as:")
print("  1. Standalone CNN classifier (loaded in 08_AllModelsCombined.ipynb)")
print("  2. Feature extractor backbone (loaded in 02_CNN_Ensemble.ipynb)")

## Section 9: Results Summary

In [ ]:
print("=" * 60)
print(f"NOTEBOOK: {NOTEBOOK_NAME}")
print(f"Dataset  : {train_generator.samples + val_generator.samples} training images")
print(f"Classes  : {CLASS_NAMES}")
print(f"Image size: {IMG_HEIGHT}\u00d7{IMG_WIDTH}")
print(f"Batch size: {BATCH_SIZE}, Epochs: {EPOCHS}, LR: {LEARNING_RATE}")
print(f"Seed     : {RANDOM_SEED}")
print("-" * 60)
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Loss     : {test_loss:.4f}")
print("=" * 60)
print("Saved models location:", SAVED_MODELS_DIR)

## Section 10: HuggingFace Upload

Uploads the model file(s) saved in Section 8 and all result JPGs from Section 7 to `shehank98/brain-tumor-mri-models` on HuggingFace Hub.

Requires `HF_TOKEN` to be set (via Colab Secrets in Section 0, or `HF_TOKEN` env var).

In [ ]:
# ── HuggingFace repository ────────────────────────────────────────────────────
HF_REPO_ID = "shehank98/brain-tumor-mri-models"   # your HF repo

try:
    from huggingface_hub import HfApi, login as hf_login
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', '-q'])
    from huggingface_hub import HfApi, login as hf_login

if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set — skipping HuggingFace upload.")
    print("  Set it in Colab Secrets (key: HF_TOKEN) or as an env var.")
else:
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    api = HfApi()

    # Create repo if it does not exist yet
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model",
                    private=False, exist_ok=True)
    print(f"✓ Repository ready: https://huggingface.co/{HF_REPO_ID}")

    # Upload model files
    _model_files = ['cnn_model.h5']
    for _fname in _model_files:
        _local = os.path.join(SAVED_MODELS_DIR, _fname)
        if not os.path.exists(_local):
            print(f"  SKIP (not found): {_fname}")
            continue
        _size = os.path.getsize(_local) / 1e6
        print(f"  Uploading {_fname} ({_size:.1f} MB)...", end="", flush=True)
        api.upload_file(
            path_or_fileobj=_local,
            path_in_repo=f"models/{_fname}",
            repo_id=HF_REPO_ID,
            commit_message=f"Upload {_fname} from 01_CNN",
        )
        print(" done")

    # Upload results JPGs
    _results_nb = os.path.join(RESULTS_DIR, NOTEBOOK_NAME)
    if os.path.exists(_results_nb):
        _jpgs = [f for f in os.listdir(_results_nb) if f.endswith('.jpg')]
        for _jpg in sorted(_jpgs):
            print(f"  Uploading result chart {_jpg}...", end="", flush=True)
            api.upload_file(
                path_or_fileobj=os.path.join(_results_nb, _jpg),
                path_in_repo=f"results/{NOTEBOOK_NAME}/{_jpg}",
                repo_id=HF_REPO_ID,
                commit_message=f"Add result chart {_jpg}",
            )
            print(" done")
        print(f"✓ {len(_jpgs)} result charts uploaded")
    else:
        print("  No result charts found — run Section 7 first")

    print(f"\n✓ Upload complete: https://huggingface.co/{HF_REPO_ID}")